In [1]:
import pandas as pd
import numpy as np

# 1️⃣ Identifiy & Explore Issues

In [2]:
df = pd.read_csv("railway.csv")

In [3]:
df.head()

,Transaction ID,Date of Purchase,Time of Purchase,Purchase Type,Payment Method,Railcard,Ticket Class,Ticket Type,Price,Departure Station,Arrival Destination,Date of Journey,Departure Time,Arrival Time,Actual Arrival Time,Journey Status,Reason for Delay,Refund Request
0,da8a6ba8-b3dc-4677-b176,2023-12-08,12:41:11,Online,Contactless,Adult,Standard,Advance,43,London Paddington,Liverpool Lime Street,2024-01-01,11:00:00,13:30:00,13:30:00,On Time,NaN,No
1,b0cdd1b0-f214-4197-be53,2023-12-16,11:23:01,Station,Credit Card,Adult,Standard,Advance,23,London Kings Cross,York,2024-01-01,09:45:00,11:35:00,11:40:00,Delayed,Signal Failure,No
2,f3ba7a96-f713-40d9-9629,2023-12-19,19:51:27,Online,Credit Card,NaN,Standard,Advance,3,Liverpool Lime Street,Manchester Piccadilly,2024-01-02,18:15:00,18:45:00,18:45:00,On Time,NaN,No
3,b2471f11-4fe7-4c87-8ab4,2023-12-20,23:00:36,Station,Credit Card,NaN,Standard,Advance,13,London Paddington,Reading,2024-01-01,21:30:00,22:30:00,22:30:00,On Time,NaN,No
4,2be00b45-0762-485e-a7a3,2023-12-27,18:22:56,Online,Contactless,NaN,Standard,Advance,76,Liverpool Lime Street,London Euston,2024-01-01,16:45:00,19:00:00,19:00:00,On Time,NaN,No


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31653 entries, 0 to 31652
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Transaction ID       31653 non-null  object
 1   Date of Purchase     31653 non-null  object
 2   Time of Purchase     31653 non-null  object
 3   Purchase Type        31653 non-null  object
 4   Payment Method       31653 non-null  object
 5   Railcard             10735 non-null  object
 6   Ticket Class         31653 non-null  object
 7   Ticket Type          31653 non-null  object
 8   Price                31653 non-null  int64 
 9   Departure Station    31653 non-null  object
 10  Arrival Destination  31653 non-null  object
 11  Date of Journey      31653 non-null  object
 12  Departure Time       31653 non-null  object
 13  Arrival Time         31653 non-null  object
 14  Actual Arrival Time  29773 non-null  object
 15  Journey Status       31653 non-null  object
 16  Reas

In [5]:
df["Railcard"].isna().sum()

np.int64(20918)

ISUEE 01: There are 20918 missing vaules at Railcard

ISSUE 02: Acual Arrive time has missing values

In [6]:
df["Reason for Delay"].unique()

array([nan, 'Signal Failure', 'Technical Issue', 'Weather Conditions',
       'Weather', 'Staffing', 'Staff Shortage', 'Signal failure',
       'Traffic'], dtype=object)

ISSUE 03: Same issue has many formats like `Signal Failure` and `Signal failure`.

ISSUE 04: Data types are not ajusted

In [7]:
df['Actual Arrival Time'].isna().sum()

np.int64(1880)

ISSUE 05: Acual Arrival Time has missing values

# 2️⃣ Data Cleaning

## Data Type Issues
- `Date of Purchase` -> Date
- `Date of Journey` -> Date
- `Time of Purchase` -> Time
- `Departure Time` -> Time
- `Arrival Time` -> Time
- `Actual Arrival Time` -> Time

In [8]:
# Convert Date datatypes
df['Date of Purchase'] = pd.to_datetime(df['Date of Purchase'], errors='coerce')
df['Date of Journey'] = pd.to_datetime(df['Date of Journey'], errors='coerce')

In [9]:
# Convert to Time
time_cols = ['Time of Purchase', 'Departure Time', 'Arrival Time', 'Actual Arrival Time']

for col in time_cols:
    df[col] = pd.to_timedelta(df[col])

In [10]:
df["Time of Purchase"]

0       0 days 12:41:11
1       0 days 11:23:01
2       0 days 19:51:27
3       0 days 23:00:36
4       0 days 18:22:56
              ...      
31648   0 days 18:42:58
31649   0 days 18:46:10
31650   0 days 18:56:41
31651   0 days 19:51:47
31652   0 days 20:05:39
Name: Time of Purchase, Length: 31653, dtype: timedelta64[ns]

## Format Issues
- `Reason for delay`, have multiple format issue

In [11]:
df['Reason for Delay'] = df['Reason for Delay'].str.strip().str.title()

df['Reason for Delay'] = df['Reason for Delay'].replace({
    'Weather': 'Weather Conditions',
    'Staffing': 'Staff Shortage'
})

df['Reason for Delay'] = df['Reason for Delay'].fillna("No Delay")

# Verify the changes
print(df['Reason for Delay'].unique())

['No Delay' 'Signal Failure' 'Technical Issue' 'Weather Conditions'
 'Staff Shortage' 'Traffic']


In [12]:
df['Railcard'] = df['Railcard'].fillna("No Railcard")

# Verify the changes
print(df['Railcard'].unique())

['Adult' 'No Railcard' 'Disabled' 'Senior']


# 3️⃣ Columns to be added

## Reveune:
- discount from rail card
- discount from ticket type
- Total Discount %
- Price before discount

## Operations:
- delay in minutes
- difference between departure time and arrive time in minutes
- `Route` route/line name (start station + end station)
- lead time (time between ticket purchase and journey date)

In [13]:
df["Journey Duration"] = (df["Arrival Time"] - df["Departure Time"]).dt.total_seconds() / 60

In [14]:
df["Route"] = df["Departure Station"] + " to " + df["Arrival Destination"]

In [15]:
df["Lead Time"] = (df["Date of Journey"] - df["Date of Purchase"]).dt.days

In [16]:
# Calculate delay in minutes
df['Delay in Minutes'] = (df['Actual Arrival Time'] - df['Arrival Time']).dt.total_seconds() / 60


Revenue Columns

In [17]:
# Ticket Types
ticket_type_discount_map = {
    "Anytime": 0.00,
    "Off-Peak": 0.25,
    "Advance": 0.50
}

df["Ticket Type Discount"] = df["Ticket Type"].map(ticket_type_discount_map)

railcard_discount_map = {
    "Adult": 1/3,
    "Senior": 1/3,
    "Disabled": 1/3
}

df["Railcard Discount"] = df["Railcard"].map(railcard_discount_map).fillna(0)

In [18]:
# Final Discount Factor Equation
Final_Discount_Factor = (
    (1 - df["Ticket Type Discount"]) *
    (1 - df["Railcard Discount"])
)

# Price before discount
df["Price before discount"] = df["Price"] / Final_Discount_Factor

df["Price before discount"] = df["Price before discount"].round(2)

In [19]:
# Total Discount %
df["Total Discount %"] = (1 - Final_Discount_Factor)

In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31653 entries, 0 to 31652
Data columns (total 26 columns):
 #   Column                 Non-Null Count  Dtype          
---  ------                 --------------  -----          
 0   Transaction ID         31653 non-null  object         
 1   Date of Purchase       31653 non-null  datetime64[ns] 
 2   Time of Purchase       31653 non-null  timedelta64[ns]
 3   Purchase Type          31653 non-null  object         
 4   Payment Method         31653 non-null  object         
 5   Railcard               31653 non-null  object         
 6   Ticket Class           31653 non-null  object         
 7   Ticket Type            31653 non-null  object         
 8   Price                  31653 non-null  int64          
 9   Departure Station      31653 non-null  object         
 10  Arrival Destination    31653 non-null  object         
 11  Date of Journey        31653 non-null  datetime64[ns] 
 12  Departure Time         31653 non-null  timedel

In [21]:
[Final_Discount_Factor, df["Total Discount %"]]

[0        0.333333
 1        0.333333
 2        0.500000
 3        0.500000
 4        0.500000
            ...   
 31648    0.750000
 31649    0.750000
 31650    0.750000
 31651    0.750000
 31652    0.500000
 Length: 31653, dtype: float64,
 0        0.666667
 1        0.666667
 2        0.500000
 3        0.500000
 4        0.500000
            ...   
 31648    0.250000
 31649    0.250000
 31650    0.250000
 31651    0.250000
 31652    0.500000
 Name: Total Discount %, Length: 31653, dtype: float64]

In [22]:
for col in time_cols:
    df[col] = df[col].astype(str).str.replace("0 days ", "", regex=False)

In [23]:
df

,Transaction ID,Date of Purchase,Time of Purchase,Purchase Type,Payment Method,Railcard,Ticket Class,Ticket Type,Price,Departure Station,...,Reason for Delay,Refund Request,Journey Duration,Route,Lead Time,Delay in Minutes,Ticket Type Discount,Railcard Discount,Price before discount,Total Discount %
0,da8a6ba8-b3dc-4677-b176,2023-12-08,12:41:11,Online,Contactless,Adult,Standard,Advance,43,London Paddington,...,No Delay,No,150.0,London Paddington to Liverpool Lime Street,24,0.0,0.50,0.333333,129.00,0.666667
1,b0cdd1b0-f214-4197-be53,2023-12-16,11:23:01,Station,Credit Card,Adult,Standard,Advance,23,London Kings Cross,...,Signal Failure,No,110.0,London Kings Cross to York,16,5.0,0.50,0.333333,69.00,0.666667
2,f3ba7a96-f713-40d9-9629,2023-12-19,19:51:27,Online,Credit Card,No Railcard,Standard,Advance,3,Liverpool Lime Street,...,No Delay,No,30.0,Liverpool Lime Street to Manchester Piccadilly,14,0.0,0.50,0.000000,6.00,0.500000
3,b2471f11-4fe7-4c87-8ab4,2023-12-20,23:00:36,Station,Credit Card,No Railcard,Standard,Advance,13,London Paddington,...,No Delay,No,60.0,London Paddington to Reading,12,0.0,0.50,0.000000,26.00,0.500000
4,2be00b45-0762-485e-a7a3,2023-12-27,18:22:56,Online,Contactless,No Railcard,Standard,Advance,76,Liverpool Lime Street,...,No Delay,No,135.0,Liverpool Lime Street to London Euston,5,0.0,0.50,0.000000,152.00,0.500000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
31648,1304623d-b8b7-4999-8e9c,2024-04-30,18:42:58,Online,Credit Card,No Railcard,Standard,Off-Peak,4,Manchester Piccadilly,...,No Delay,No,30.0,Manchester Piccadilly to Liverpool Lime Street,0,0.0,0.25,0.000000,5.33,0.250000
31649,7da22246-f480-417c-bc2f,2024-04-30,18:46:10,Online,Contactless,No Railcard,Standard,Off-Peak,10,London Euston,...,No Delay,No,80.0,London Euston to Birmingham New Street,0,0.0,0.25,0.000000,13.33,0.250000
31650,add9debf-46c1-4c75-b52d,2024-04-30,18:56:41,Station,Credit Card,No Railcard,Standard,Off-Peak,4,Manchester Piccadilly,...,No Delay,No,30.0,Manchester Piccadilly to Liverpool Lime Street,0,0.0,0.25,0.000000,5.33,0.250000
31651,b92b047c-21fd-4859-966a,2024-04-30,19:51:47,Station,Credit Card,No Railcard,Standard,Off-Peak,10,London Euston,...,No Delay,No,80.0,London Euston to Birmingham New Street,0,0.0,0.25,0.000000,13.33,0.250000


In [24]:
df.to_csv("cleaned_railway.csv", index=False)